# EDA — SpaceX Launch Success

Análise exploratória do dataset de lançamentos. Reutiliza os módulos do
pacote `launch_success` (mesma limpeza/engenharia do pipeline), evitando
divergência entre o notebook e a produção.

> Rode `make install` (ou `pip install -e .`) antes de abrir o notebook.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from launch_success.config import SETTINGS
from launch_success.data.loader import load_dataset
from launch_success.features.cleaning import clean_launches
from launch_success.features.engineering import add_derived_features

sns.set_theme(style="whitegrid")
TARGET = SETTINGS.target

raw = load_dataset(settings=SETTINGS)
df = add_derived_features(clean_launches(raw, settings=SETTINGS))
print(df.shape)
df.head()

## 1. Distribuição do alvo (desbalanceamento)

In [ ]:
print(df[TARGET].value_counts(normalize=True).round(3))
sns.countplot(x=df[TARGET])
plt.title(f"Distribuição do alvo '{TARGET}'")
plt.show()

## 2. Taxa de sucesso por foguete, órbita e reúso

In [ ]:
for col in ["rocket", "orbit", "reused"]:
    display(df.groupby(col)[TARGET].mean().sort_values(ascending=False).round(3))

## 3. Massa do payload por órbita e correlações numéricas

In [ ]:
sns.boxplot(data=df, x="orbit", y="payload_mass_kg")
plt.title("Massa do payload por órbita")
plt.xticks(rotation=30)
plt.show()

num_cols = ["flight_number", "year", "payload_mass_kg", "flights", TARGET]
sns.heatmap(df[num_cols].corr(numeric_only=True), annot=True, cmap="coolwarm", center=0)
plt.title("Correlação entre variáveis numéricas e o alvo")
plt.show()

## Conclusões

- O alvo é **desbalanceado** (sucesso > 95%), exigindo métricas além da acurácia.
- **Falcon 1** concentra as falhas; **Falcon 9/Heavy** são altamente confiáveis.
- `flight_number`/`year` (maturação) e `payload_mass_kg` são os sinais mais
  fortes — confirmado pela análise SHAP do pipeline (`make train`).